In [ ]:
# general imports 
import numpy as np
import os
import matplotlib.pyplot as plt
import math
import pandas as pd
import MDAnalysis as mda
from IPython.display import Video

# Today's Practical

- We will construct a multicomponent system composed of a membrane, peptide, water, and ions.
- The main objective of today's practical is to learn which steps can/should be taken for efficient system assembly.

In [ ]:
!ls -l

## What is inside?

### force_field

You know what it is. CHARMM force field, similarly to the previous practical.

### images

Images related to the practical.

### scripts

Contains `.mdp` files, including those with defined position restraints.

### structures

Starting atomic configurations of molecules of interest.

### videos

Videos related to the practical.

### trajectories

Contains already conducted simulations for the purposes of today's practical.

### week9.ipynb

Jupyter notebook we are using right now.

### Create a folder where we will be working today

In [ ]:
!mkdir playground/

In [ ]:
%cd playground/

## Peptide adsorption to the membrane

Why peptides?

- antimicrobial agents
- drug delivery design
- viral fusion
- biosensors
- more...

## Preparing the system. What do we need?

- Membrane (previous week, CHARMM-GUI)
- Peptide (will learn today)
- Solvent (first practical with Jupyter notebook)
- Ions (second practical with Jupyter notebook)

### What do we already have?

In [ ]:
!ls -l ../structures/

## Membrane

### What is the membrane structure at the moment?

- 200 POPC lipids (the most common lipid studied)
- 10,000 water moleculess (50 per lipid, standard hydration conditions)

<img src='../../images/membrane_water.png' width=700 height=700 alt='membrane_water'>

### What do we do?

- The most typical output from builders &ndash; hydrated lipid bilayer.
- In order to have a peptide/protein too, you should either:
    - Ask builder to add it.
    - If not possible, initially ask builder to add more water (i.e., more available space) to add peptide/protein later, replacing water.
    - Another option: get rid of water, make a larger box, add peptide, and then add as much water as needed (our choice today for more experience).

### Getting rid of water

In [ ]:
!cp -r ../structures/membrane.gro .

In [ ]:
!head -n 10 membrane.gro

In [ ]:
!tail -n 10 membrane.gro

### We want to remove all the water

- We will use Linux functionality for it.

In [ ]:
!grep -v 'SOL' membrane.gro > membrane_nowater.gro

In [ ]:
!head -n 10 membrane_nowater.gro

In [ ]:
!tail -n 10 membrane_nowater.gro

### We also want to change the total number of atoms

- Another Linux functionality.

In [ ]:
!sed -i '2s/.*/26800/' membrane_nowater.gro

In [ ]:
!head -n 10 membrane_nowater.gro

In [ ]:
!tail -n 10 membrane_nowater.gro

<img src='../../images/membrane_nowater.png' width=700 height=700 alt='membrane_nowater'>

## Our peptide

### Magainin 2

- Antimicrobial peptide
- 23 amino acid residues
- Originally isolated from the African clawed frog (*Xenopus laevis*)
- Often serves as a model for studying host&mdash;defense peptides
- Forms an amphipathic $\alpha$-helix in membranes

### Structure from pdb as an example

- For $\alpha$-helical peptides, PDB database is not necessary.
- Any building tool should work.

<img src='../../images/peptide_pdbpage.png' width=1200 height=900 alt='peptide_pdb'>

### Peptide topology

- Download the PDB structure.
- Generate the topology with `gmx pdb2gmx`.
- Alternatively, use CHARMM-GUI (still requires PDB file).

In [ ]:
!wget https://files.rcsb.org/download/2LSA.pdb

### Additional tricks

- The downloaded PDB contains 10 structures &ndash; we need only one
- Remove all structures except the first one

In [ ]:
!sed '/^ENDMDL/q' 2LSA.pdb > 2LSA_single.pdb

<img src='../../images/peptide_pdb.png' width=700 height=700 alt='peptide_pdb'>

### `gmx pdb2gmx`

- `gmx pdb2gmx` reads a `.pdb` (or `.gro`) file and generates coordinates/topologies in GROMACS format.
- `-f` &mdash; input `.pdb` structure
- `-o` &mdash; name of the output `.gro` (or `.pdb`) structure
- `-p` &mdash; name of the output topology file (`.top`, `.itp`)
- `-i` &mdash; name of the separate output restraint file (`.itp`)
- `-[no]ignh` &mdash; ignore hydrogen atoms in the original structure (highly recommended)
- `-water` &mdash; water model to use; we omit this option at this stage, because we are not interested in possible crystal water
- there are many other options that we will not cover today; check `gmx pdb2gmx -h` or https://manual.gromacs.org/current/onlinehelp/gmx-pdb2gmx.html yourself

### Force field for topology generation

- GROMACS provides several built-in force fields (often older versions).
- Alternatively, you can use GROMACS-compatible force field files placed in the same directory, where you have the structure (our option).

In [ ]:
!cp -r ../force_field/charmm36-jul2022.ff/ .

In [ ]:
input_command = """1
"""
!echo "$input_command" | gmx pdb2gmx -f 2LSA_single.pdb -o peptide.gro -p peptide.top -i peptide_restr.itp -ignh -water none

## What did we generate?

In [ ]:
!ls -l

## We should adjust the topology files

- GROMACS is not aware we intend to build a membrane–peptide system.
- The generated files are configured for a peptide-only simulation and can run as is.

In [ ]:
!head -n 40 peptide.top

In [ ]:
!tail -n 40 peptide.top

### Adjusting

- Deleting everything before [ moleculetype ].
- Delete everything after [ system ].

In [ ]:
!sed -n '/\[ moleculetype \]/,/\[ system \]/p' peptide.top | sed '$d' > peptide.itp

In [ ]:
!head -n 20 peptide.itp

In [ ]:
!tail -n 20 peptide.itp

### What is a restraint file?

- List of atoms to be restrained. 
- Only heavy atoms.
- Force constant is defined.

In [ ]:
!head -n 20 peptide_restr.itp

In [ ]:
!tail -n 20 peptide_restr.itp

## Selecting the box size and putting both membrane and peptide there

### Box size

- The simulation box must be large enough to capture all possible peptide behaviors &mdash; the peptide is far from the membrane, approaching it, or fully adsorbed onto its surface.
- We will use already known `gmx editconf` command.

In [ ]:
!tail -n 5 membrane_nowater.gro

In [ ]:
!gmx_mpi editconf -f membrane_nowater.gro -o membrane_nowater.gro -box 8.06145 8.06145 14 -c

### What have we done?

- We increased the box size in z-direction, i.e., along the membrane normal.

<img src='../images/membrane_box.png' width=700 height=700 alt='membrane_box'>

### Consistent box size for peptide?

In [ ]:
!gmx_mpi editconf -f peptide.gro -o peptide.gro -box 8.06145 8.06145 14 -c

In [ ]:
!rm -f *#*

<img src='../images/peptide_box.png' width=700 height=700 alt='peptide_box'>

## Merging structures (membrane and peptide) into one using Linux tools

- Take all lines of the membrane box except the last one.
- Take all lines of the peptide except the first two.
- Update the total number of atoms on the second line.
- Update the box size (if necessary).

In [ ]:
!head -n 3 membrane_nowater.gro

In [ ]:
!head -n 3 peptide.gro

In [ ]:
!{ head -n -1 membrane_nowater.gro; tail -n +3 peptide.gro; } | awk 'NR==1 {print "membrane and peptide"}; NR!=1' | awk 'NR==2 {print "27157"}; NR!=2' > membrane_peptide.gro

In [ ]:
!head -n 10 membrane_peptide.gro

In [ ]:
!tail -n 10 membrane_peptide.gro

<img src='../images/membrane_peptide_wrong.png' width=700 height=700 alt='membrane_peptide_wrong'>

### We need to fix the system

- Shift the coordinated of peptides before the merging to avoid an overlap.
- Already known `gmx editconf` command.

In [ ]:
!gmx editconf -f peptide.gro -o peptide_shifted.gro -translate 0 0 4

In [ ]:
!{ head -n -1 membrane_nowater.gro; tail -n +3 peptide_shifted.gro; } | awk 'NR==1 {print "membrane and peptide"}; NR!=1' | awk 'NR==2 {print "27157"}; NR!=2' > membrane_peptide.gro

<img src='../images/membrane_peptide.png' width=700 height=700 alt='membrane_peptide'>

## Solvating the system

- Already known `gmx solvate` command.

## TASK

- Solvate the system.
- Number of water molecules should not exceed 21,000.
- The output file should be named as `membrane_peptide_water.gro`.

## Expected result:

<img src='../images/membrane_peptide_water.png' width=700 height=700 alt='membrane_peptide_water'>

<img src='../images/membrane_peptide_water_just_water.png' width=700 height=700 alt='membrane_peptide_water_just_water'>

## Adding ions

Two possible approaches:

- `gmx insert-molecules` &mdash; we are already familiar with it.
- `gmx genion` &mdash; requires generating a binary `.tpr` file beforehand.
- If we need a binary, we need to create a complete `topol.top` file first.

In [ ]:
topology = """; force field parameters
#include "../force_field/charmm36-jul2022.ff/forcefield.itp"
#include "../force_field/charmm36-jul2022.ff/tip3p.itp"
#include "../force_field/charmm36-jul2022.ff/ions.itp"
#include "./peptide.itp"
#include "../force_field/POPC.itp"

[ system ]
; name of the system
membrane-peptide-water system

[ molecules ]
; compound   #mols
POPC               200
Protein_chain_A    1
SOL                21000
"""

with open('topol.top', 'w') as outFile:
    outFile.write(topology)

### Important notes
- The order of `include` **is important**
- The order of molecules **is important**

Let us also have a look at `POPC.itp`

In [ ]:
!head -n 20 ../force_field/POPC.itp

In [ ]:
!tail -n 20 ../force_field/POPC.itp

- Restraints are a part of the same file (possible in GROMACS).
- Both options are fine.

### Generating `.tpr` using `gmx grompp`

- You already know the basics.
- New flag is `-r` &mdash; supplies a reference structure for position restraints (later about it).

In [ ]:
!gmx grompp -p topol.top -c membrane_peptide_water.gro -r membrane_peptide_water.gro -f ../scripts/minim.mdp -o forgenion.tpr -po forgenion_mdout.mdp -maxwarn 1

### `gmx genion`

- `gmx genion` randomly replaces solvent molecules with monoatomic ions.
- `-s` &mdash; input `.tpr` file
- `-p` &mdash; name of the topology `.top` file
- `-o` &mdash; name of the output structure (`.gro`)
- `-np` &mdash; number of positively charged ions
- `-nn` &mdash; number of negatively charged ions
- `-pname` &mdash; name of a positively charged ion (default: NA)
- `-nname` &mdash; name of a negatively charged ion (default: CL)
- `-conc` &mdash; adds ions to yield a given salt concentration (molarity; in M); overrides `-np` and `-nn`
- `-[no]neutral` &mdash; add enough ions to neutralize the system; on top of added ions by other keywords
- there are many other options that we will not cover today; check `gmx genion -h` or https://manual.gromacs.org/current/onlinehelp/gmx-genion.html yourself

In [ ]:
input_command = """SOL
"""
!echo "$input_command" | gmx genion -s forgenion.tpr -p topol.top -o membrane_peptide_water_ions.gro -conc 0.15 -neutral 

<img src='../images/membrane_peptide_water_ions.png' width=700 height=700 alt='membrane_peptide_water_ions'>

## Is it the correct ionic concentration?

- What GROMACS does not consider?
- How to fix it?
- Let's restore the original `topol.top` file

In [ ]:
!mv \#topol.top.1# topol.top

## TASK

- Calculate how many ions should be added.
- Replace XX with proper numbers.

In [ ]:
# input_command = """SOL
# """
# !echo "$input_command" | gmx genion -s forgenion.tpr -p topol.top -o membrane_peptide_water_ions.gro -np XX -nn XX -neutral 

## How restraints are enabled in a simulation?

- Let's have a look at the topology files again.

In [ ]:
!tail -n 10 peptide.itp

In [ ]:
!head -n 20 peptide_restr.itp

In [ ]:
!tail -n 15 ../force_field/POPC.itp

## Restraints are conditionally enabled in `.mdp` files

- Controlled through the `define` directive.
- Can be switched on or off, with optional specification of the force constant.

In [ ]:
!head -n 5 ../scripts/minim.mdp

In [ ]:
!head -n 5 ../scripts/eq.mdp

### Minimization and equilibration

In [ ]:
# !gmx grompp -p topol.top -c membrane_peptide_water_ions.gro -r membrane_peptide_water_ions.gro -f ../scripts/minim.mdp -o minim.tpr -po minim_mdout.mdp

In [ ]:
# !gmx mdrun -deffnm minim -v

In [ ]:
# !gmx grompp -p topol.top -c minim.gro -r membrane_peptide_water_ions.gro -f ../scripts/eq.mdp -o eq.tpr -po eq_mdout.mdp -n index.ndx -maxwarn 2

In [ ]:
# !gmx mdrun -deffnm eq -v

### These simulations are a bit costly
- We will not run these simulations now (commands are commented).
- You would also need to create index groups for a separate thermostatting &mdash; `MEMB`, `SOLV`, `PROT`.
- Pay attention to `-r` keyword.
- The simulations were already performed for you.
- Let's copy the trajectories to the current directory.

In [ ]:
!cp ../trajectories/* .

In [ ]:
Video('../videos/equilibration_system.mp4', width=800, height=800, embed=True)

In [ ]:
Video('../videos/equilibration_water.mp4', width=800, height=800, embed=True)

### Equilibration

- Membrane and peptide were successfully restrained.
- Water escaped the hydrophobic core of the membrane.

### TASK
- Perform any of the previous analyses, such as potential energy during minimization, temperature or pressure during equilibration, or other relevant system properties (for instance, density).
- Plot and explain the results.